In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

c:\rag\.venv\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (6.0.0dev0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\rag\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from sentence_transformers import SentenceTransformer
from typing import List
import numpy as np
import contextlib
import os

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model without printing internal logs"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            # Suppress stdout temporarily to hide BertModel LOAD REPORT
            with open(os.devnull, "w") as f, contextlib.redirect_stdout(f):
                self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not self.model:
            raise ValueError("Model not loaded")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# Initialize embedding manager
embedding_manager = EmbeddingManager()
embedding_manager



Loading embedding model: all-MiniLM-L6-v2
Model loaded successfully. Embedding dimension: 384


In [ ]:
import os
import uuid
from typing import List, Any
import numpy as np
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "../data/vector_store"
    ):
        """
        Initialize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""

        # Create persistent ChromaDB client
        os.makedirs(self.persist_directory, exist_ok=True)
        self.client = chromadb.PersistentClient(
            path=self.persist_directory
        )

        # Get or create collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "PDF document embeddings for RAG"}
        )

        print(f"Vector store initialized. Collection: {self.collection_name}")
        print(f"Existing documents in collection: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        self.collection.add(
            ids=ids,
            embeddings=embeddings_list,
            metadatas=metadatas,
            documents=documents_text
        )

        print(f"Successfully added {len(documents)} documents to vector store")
        print(f"Total documents in collection: {self.collection.count()}")
vector_store = VectorStore()
vector_store
# Chunking Function
def create_chunks(text: str, chunk_size=500, chunk_overlap=50):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = splitter.split_text(text)

    documents = []
    for i, chunk in enumerate(chunks):
        documents.append(
            Document(
                page_content=chunk,
                metadata={"chunk_id": i}
            )
        )

    return documents


# Embedding Function
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_chunks(documents):
    texts = [doc.page_content for doc in documents]
    embeddings = embedding_model.encode(texts, show_progress_bar=True)
    return np.array(embeddings)



    # Create chunks
    documents = create_chunks(text_data)

    # Create embeddings
    embeddings = embed_chunks(documents)

    # Store in ChromaDB
    vector_store = VectorStore()
    vector_store.add_documents(documents, embeddings)

KeyError: '_type'

In [ ]:
chunks = create_chunks(text_data)
chunks


NameError: name 'text_data' is not defined

In [ ]:
# REQUIRED INSTALLS (run once)
# pip install langchain-community langchain-text-splitters pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ---------- 1. LOAD PDF ----------
pdf_path = "../data/pdf/attention.pdf"   # change path if needed
loader = PyPDFLoader(pdf_path)
documents = loader.load()   # list of Document (page-wise)


# ---------- 2. CHUNKING ----------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)


# ---------- 3. OUTPUT ----------
print("Total pages:", len(documents))
print("Total chunks:", len(chunks))

print("\nFIRST CHUNK TEXT:\n")
print(chunks[0].page_content)


Total pages: 15
Total chunks: 93

FIRST CHUNK TEXT:

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu


In [ ]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../data/pdf/attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu'),
 Document(metadata={'pr

In [ ]:
texts=[doc.page_content for doc in chunks]
texts

['Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu',
 'University of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser ∗\nGoogle Brain\nlukaszkaiser@google.com\nIllia Polosukhin∗ ‡\nillia.polosukhin@gmail.com\nAbstract\nThe dominant sequence transduction models are based on complex recurrent or\nconvolutional neural networks that include an encoder and a decoder. The best\nperforming models also connect the encoder and decoder through an attention\nmechanism. We propose a new simple network architecture, the Transformer,',
 'based sol

In [ ]:

texts = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(texts)
vector_store.add_documents(chunks, embeddings)


Generating embeddings for 93 texts...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.35it/s]

Generated embeddings with shape: (93, 384)
Adding 93 documents to vector store...
Successfully added 93 documents to vector store
Total documents in collection: 279


In [ ]:

import os
import chromadb
from sentence_transformers import SentenceTransformer
from pypdf import PdfReader



def load_pdf_text(pdf_path):
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found at path: {pdf_path}")

    reader = PdfReader(pdf_path)
    text = ""

    for page in reader.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

    return text



def chunk_text(text, chunk_size=800, overlap=100):
    chunks = []
    start = 0
    text_length = len(text)

    while start < text_length:
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks



class EmbeddingManager:
    def __init__(self):
        self.model = SentenceTransformer(
            "sentence-transformers/all-MiniLM-L6-v2"
        )

    def embed_documents(self, texts):
        return self.model.encode(texts, convert_to_numpy=True)

    def embed_query(self, query):
        return self.model.encode(query, convert_to_numpy=True)



class VectorStore:
    def __init__(self):
        self.client = chromadb.Client()
        self.collection = self.client.get_or_create_collection(
            name="pdf_rag_collection"
        )

    def add_documents(self, chunks, embeddings):
        ids = [f"chunk_{i}" for i in range(len(chunks))]
        self.collection.add(
            documents=chunks,
            embeddings=embeddings.tolist(),
            ids=ids
        )

    def count(self):
        return self.collection.count()



class RAGRetriever:
    def __init__(self, vector_store, embedding_manager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query, top_k=5, max_distance=1.2):
        query_embedding = self.embedding_manager.embed_query(query)

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k
        )

        docs = results["documents"][0]
        distances = results["distances"][0]

        return [
            doc for doc, dist in zip(docs, distances)
            if dist < max_distance
        ]



pdf_path = r"C:\rag\data\pdf\attention.pdf"

print("PDF exists:", os.path.exists(pdf_path))

print("\nLoading PDF...")
text_data = load_pdf_text(pdf_path)

if not text_data.strip():
    raise ValueError("No text found in PDF (scanned PDF needs OCR)")

# Chunking
chunks = chunk_text(text_data)
print("Total chunks:", len(chunks))

# Initialize
embedding_manager = EmbeddingManager()
vector_store = VectorStore()

# Embed & store (only once)
if vector_store.count() == 0:
    print("Embedding and storing chunks...")
    embeddings = embedding_manager.embed_documents(chunks)
    vector_store.add_documents(chunks, embeddings)

print("Stored chunks:", vector_store.count())

# RAG Retriever
rag_retriever = RAGRetriever(vector_store, embedding_manager)


while True:
    query = input("\nAsk a question from the PDF (type 'exit' to quit): ")
    if query.lower() == "exit":
        break

    results = rag_retriever.retrieve(query)

    print("\n🔍 Retrieved Chunks:\n")
    if not results:
        print("No relevant chunks found.")
    else:
        for i, r in enumerate(results, 1):
            print(f"{i}. {r[:500]}...\n")


PDF exists: True

Loading PDF...
Total chunks: 57


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 511.14it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding and storing chunks...
Stored chunks: 57


In [ ]:
def rag_simple(query, retriever, llm, top_k=3):

    results = retriever.get_relevant_documents(query)
    context = "\n\n".join([doc.page_content for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    prompt = f"""
    Use the following context to answer the question concisely.

    Context:
    {context}

    Question: {query}

    Answer:
    """

    response = llm.invoke(prompt)
    return response.content